In [ ]:
# ── 셀 0. 환경 설정 / 전역 상수 / 공통 헬퍼 함수 ──────────────
import subprocess, sys, os, glob, json, zipfile, tarfile, shutil, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm

# 한글 matplotlib 폰트
try:
    import koreanize_matplotlib
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'koreanize-matplotlib'], check=True)
    import koreanize_matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
plt.rcParams['axes.unicode_minus'] = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

# ── 전역 상수 ──────────────────────────────────────────────────
DATASET_KEY  = 7965
WORK_DIR     = '/content/sign_work'
TMP_DIR      = '/content/sign_tmp'
PROGRESS_CSV = os.path.join(WORK_DIR, 'keypoints_progress.csv')
SEQ_LEN      = 32        # 시퀀스 길이 (프레임 수)
N_KEYPOINTS  = 21        # 손 관절 수
FEATURE_DIM  = N_KEYPOINTS * 2  # 42 (x, y)
os.makedirs(WORK_DIR, exist_ok=True)

TARGET_WORDS = [
    '안녕하세요','감사합니다','죄송합니다','괜찮아요','잠깐만요',
    '좋아요','싫어요','기뻐요','슬퍼요','화나요',
    '네','아니요','모르겠어요','도와주세요','아파요',
    '밥','물','화장실','집','병원',
    '나','너','엄마','아빠','친구',
    '먹다','마시다','가다','오다','자다',
]
TARGET_SET = set(TARGET_WORDS)

WORD_CATEGORY = {}
for w in ['안녕하세요','감사합니다','죄송합니다','괜찮아요','잠깐만요']: WORD_CATEGORY[w]='인사/예절'
for w in ['좋아요','싫어요','기뻐요','슬퍼요','화나요']:                 WORD_CATEGORY[w]='감정'
for w in ['네','아니요','모르겠어요','도와주세요','아파요']:              WORD_CATEGORY[w]='응답/요청'
for w in ['밥','물','화장실','집','병원']:                               WORD_CATEGORY[w]='사물/장소'
for w in ['나','너','엄마','아빠','친구']:                               WORD_CATEGORY[w]='사람'
for w in ['먹다','마시다','가다','오다','자다']:                         WORD_CATEGORY[w]='동작'

# ── 공통 헬퍼 함수 ────────────────────────────────────────────

def extract_word_from_json(data: dict) -> str | None:
    # 다양한 키 이름을 순서대로 시도
    for key in ['word','label','sign','name','단어','수어단어']:
        if key in data:
            return str(data[key])
    for key in ['annotation','metadata','info','data']:
        if key in data and isinstance(data[key], dict):
            r = extract_word_from_json(data[key])
            if r: return r
    return None

def video_id_from_path(path: str) -> str:
    base = os.path.splitext(os.path.basename(path))[0]
    for s in ('_morpheme','_keypoint','_F','_morph'):
        if base.endswith(s): base = base[:-len(s)]
    return base

def parse_keypoints_from_frame(frame: dict) -> np.ndarray | None:
    # 오른손 우선, 없으면 왼손(x 반전으로 오른손 기준 통일)
    right = frame.get('hand_right_keypoints_2d', [])
    left  = frame.get('hand_left_keypoints_2d',  [])
    if right and len(right) == N_KEYPOINTS:
        pts = np.array([[p[0], p[1]] for p in right], dtype=np.float32)
    elif left and len(left) == N_KEYPOINTS:
        pts = np.array([[p[0], p[1]] for p in left],  dtype=np.float32)
        pts[:, 0] *= -1
    else:
        return None
    pts -= pts[0]  # 손목(0번)을 원점으로
    scale = float(np.linalg.norm(pts[9]))
    if scale > 1e-6: pts /= scale  # 중지 첫 마디 거리로 정규화
    return pts.flatten()

def frames_from_keypoint_json(data) -> list:
    if isinstance(data, dict):
        for key in ('people','frames','annotations','data'):
            if key in data and isinstance(data[key], list): return data[key]
        if 'people' in data and isinstance(data['people'], dict): return [data['people']]
    return data if isinstance(data, list) else []

def sequence_from_keypoint_file(path: str) -> np.ndarray | None:
    try:
        with open(path, encoding='utf-8') as f: data = json.load(f)
    except (OSError, json.JSONDecodeError): return None
    vecs = [v for frame in frames_from_keypoint_json(data)
            if isinstance(frame, dict)
            for v in [parse_keypoints_from_frame(frame)] if v is not None]
    if not vecs: return None
    arr = np.array(vecs, dtype=np.float32)
    if len(arr) >= SEQ_LEN:
        # 균일 샘플링으로 SEQ_LEN 길이 고정
        return arr[np.linspace(0, len(arr)-1, SEQ_LEN).astype(int)]
    # 부족한 프레임은 마지막 프레임으로 패딩
    return np.concatenate([arr, np.repeat(arr[-1:], SEQ_LEN-len(arr), axis=0)])

def combine_split_parts(search_dir: str) -> bool:
    # aihubshell 다운로드 파일은 .zip.part0, .zip.part1073741824 ... 형식으로 분할됨
    parts = glob.glob(os.path.join(search_dir, '**', '*.zip.part*'), recursive=True)
    if not parts: return False
    bases = {}
    for p in parts:
        base = p.rsplit('.part', 1)[0]
        bases.setdefault(base, []).append(p)
    for base, ps in bases.items():
        # 바이트 오프셋 순 정렬 후 합치기
        ps_sorted = sorted(ps, key=lambda x: int(x.rsplit('.part', 1)[-1]))
        with open(base, 'wb') as out:
            for p in ps_sorted:
                with open(p, 'rb') as f: out.write(f.read())
        for p in ps_sorted: os.remove(p)
    return True

def aihub_download_and_get_zips(filekey, tmp_dir: str) -> list:
    # 이미 zip이 있으면 그대로 반환 (재실행 안전)
    existing = glob.glob(os.path.join(tmp_dir, '**', '*.zip'), recursive=True)
    if existing: return existing
    if filekey is None: return []  # Drive 복사 후 호출 시
    subprocess.run(
        f'aihubshell -mode d -datasetkey {DATASET_KEY} -filekey {filekey}',
        shell=True, cwd=tmp_dir,
    )
    for tar_path in glob.glob(os.path.join(tmp_dir, '**', '*.tar'), recursive=True):
        try:
            with tarfile.open(tar_path) as t: t.extractall(tmp_dir)
            os.remove(tar_path)
        except Exception as e: print(f'  tar 오류: {e}')
    combine_split_parts(tmp_dir)
    return glob.glob(os.path.join(tmp_dir, '**', '*.zip'), recursive=True)

def process_keypoint_dir(extract_dir: str, part_id: str,
                         word_map: dict, records: list, seq_store: str) -> int:
    files = glob.glob(os.path.join(extract_dir, '**', '*.json'), recursive=True)
    added = 0
    for path in tqdm(files, desc=f'[{part_id}]', leave=False):
        vid  = video_id_from_path(path)
        word = word_map.get(vid)
        if word is None or word not in TARGET_SET: continue
        seq = sequence_from_keypoint_file(path)
        if seq is None: continue
        sample_id = f'{part_id}_{added}'
        np.save(os.path.join(seq_store, f'{sample_id}.npy'), seq)
        records.append({'sample_id': sample_id, 'video_id': vid,
                        'word': word, 'part': part_id})
        added += 1
    return added

print('헬퍼 함수 로드 완료')

In [ ]:
# ── 셀 1. aihubshell 설치 및 인증 ─────────────────────────────
import os, subprocess

if not os.path.exists('/usr/bin/aihubshell'):
    subprocess.run(
        'curl -o /usr/bin/aihubshell https://api.aihub.or.kr/api/aihubshell.do',
        shell=True, check=True,
    )
    subprocess.run('chmod +x /usr/bin/aihubshell', shell=True, check=True)

# ↓ 본인 계정으로 교체
AIHUB_ID = 'your_id@example.com'
AIHUB_PW = 'your_password'

os.environ['aihubid'] = AIHUB_ID
os.environ['aihubpw'] = AIHUB_PW

result = subprocess.run(
    f'aihubshell -mode l -id {AIHUB_ID} -pw {AIHUB_PW}',
    shell=True, capture_output=True, text=True,
)
print(result.stdout[-500:] or '(출력 없음)')
if result.stderr: print('STDERR:', result.stderr[-200:])

In [ ]:
# ── 셀 2. Google Drive 마운트 + REAL/WORD 형태소 파싱 ──────────
import os, glob, json, zipfile, shutil
from google.colab import drive
from tqdm.auto import tqdm

drive.mount('/content/drive')
DRIVE_DIR    = '/content/drive/MyDrive'
MORPHEME_ZIP = os.path.join(DRIVE_DIR, '01_real_word_morpheme.zip')
MORPHEME_DIR = '/content/real_morpheme'

if not os.path.exists(MORPHEME_ZIP):
    raise FileNotFoundError(f'Drive에 01_real_word_morpheme.zip 을 올려주세요.')

if os.path.exists(MORPHEME_DIR): shutil.rmtree(MORPHEME_DIR)
os.makedirs(MORPHEME_DIR, exist_ok=True)
with zipfile.ZipFile(MORPHEME_ZIP) as zf: zf.extractall(MORPHEME_DIR)

morpheme_files = glob.glob(os.path.join(MORPHEME_DIR, '**', '*.json'), recursive=True)
print(f'형태소 JSON {len(morpheme_files):,}개')

video_to_word, fail = {}, 0
for path in tqdm(morpheme_files, desc='REAL 형태소'):
    try:
        with open(path, encoding='utf-8') as f: data = json.load(f)
    except: fail += 1; continue
    word = extract_word_from_json(data)
    if word is None: fail += 1; continue
    video_to_word[video_id_from_path(path)] = word.strip()

target_cnt = sum(1 for w in video_to_word.values() if w in TARGET_SET)
print(f'매핑 {len(video_to_word):,}개 | 대상 {target_cnt:,}개 | 실패 {fail}')

In [ ]:
# ── 셀 3. REAL/WORD 키포인트 01~16 순차 처리 ──────────────────
import os, glob, zipfile, shutil
import numpy as np
import pandas as pd

# filekey: AI Hub 파일 식별자
REAL_WORD_FILEKEYS = {
    1:39600, 2:39602, 3:39603, 4:39604,  5:39605,
    6:39606, 7:39607, 8:39608, 9:39609, 10:39610,
    11:39611,12:39612,13:39613,14:39614, 15:39615, 16:39616,
}

SEQ_STORE = os.path.join(WORK_DIR, 'sequences')
os.makedirs(SEQ_STORE, exist_ok=True)

# 중간 저장 복원 (런타임 끊김 대비)
if os.path.exists(PROGRESS_CSV):
    _df = pd.read_csv(PROGRESS_CSV)
    records    = _df.to_dict('records')
    done_parts = set(_df['part'].unique())
    print(f'이어서 진행: {len(records):,}개, 완료={sorted(done_parts)}')
else:
    records, done_parts = [], set()

for num, filekey in REAL_WORD_FILEKEYS.items():
    part_id = f'rw_{num}'
    if part_id in done_parts: print(f'  {part_id} 완료 — 건너뜀'); continue

    print(f'\n[{part_id}] filekey={filekey}')
    if os.path.exists(TMP_DIR): shutil.rmtree(TMP_DIR)
    os.makedirs(TMP_DIR, exist_ok=True)

    if num == 1:
        # 1번만 Drive에서 복사 (이미 업로드됨)
        src = os.path.join(DRIVE_DIR, '01_real_word_keypoint.zip')
        if not os.path.exists(src): print(f'  {src} 없음 — 건너뜀'); continue
        shutil.copy(src, os.path.join(TMP_DIR, '01_real_word_keypoint.zip'))
        zip_paths = aihub_download_and_get_zips(None, TMP_DIR)
    else:
        zip_paths = aihub_download_and_get_zips(filekey, TMP_DIR)

    if not zip_paths: print('  zip 없음 — 건너뜀'); continue

    extract_dir = os.path.join(TMP_DIR, 'extract')
    os.makedirs(extract_dir, exist_ok=True)
    for zp in zip_paths:
        try:
            with zipfile.ZipFile(zp) as zf: zf.extractall(extract_dir)
        except zipfile.BadZipFile: print(f'  손상된 zip: {zp}')

    added = process_keypoint_dir(extract_dir, part_id, video_to_word, records, SEQ_STORE)
    done_parts.add(part_id)
    print(f'  +{added:,}개 (누적 {len(records):,})')
    pd.DataFrame(records).to_csv(PROGRESS_CSV, index=False)
    shutil.rmtree(TMP_DIR, ignore_errors=True)

print(f'\nREAL/WORD 완료: {len(records):,}개')

In [ ]:
# ── 셀 4. CROWD 형태소 파싱 ────────────────────────────────────
import os, glob, json, zipfile, shutil
from tqdm.auto import tqdm

CROWD_MORPHEME_ZIP = os.path.join(DRIVE_DIR, '01_crowd_morpheme.zip')
CROWD_MORPHEME_DIR = '/content/crowd_morpheme'

if not os.path.exists(CROWD_MORPHEME_ZIP):
    raise FileNotFoundError('Drive에 01_crowd_morpheme.zip 을 올려주세요.')

if os.path.exists(CROWD_MORPHEME_DIR): shutil.rmtree(CROWD_MORPHEME_DIR)
os.makedirs(CROWD_MORPHEME_DIR, exist_ok=True)
with zipfile.ZipFile(CROWD_MORPHEME_ZIP) as zf: zf.extractall(CROWD_MORPHEME_DIR)

crowd_files = glob.glob(os.path.join(CROWD_MORPHEME_DIR, '**', '*.json'), recursive=True)

crowd_video_to_word, fail = {}, 0
for path in tqdm(crowd_files, desc='CROWD 형태소'):
    try:
        with open(path, encoding='utf-8') as f: data = json.load(f)
    except: fail += 1; continue
    word = extract_word_from_json(data)
    if word is None: fail += 1; continue
    crowd_video_to_word[video_id_from_path(path)] = word.strip()

target_cnt = sum(1 for w in crowd_video_to_word.values() if w in TARGET_SET)
print(f'CROWD 매핑 {len(crowd_video_to_word):,}개 | 대상 {target_cnt:,}개 | 실패 {fail}')

In [ ]:
# ── 셀 5. CROWD + SYN 키포인트 처리 ───────────────────────────
import os, glob, json, zipfile, shutil
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# 런타임 재시작 시 records/done_parts 복원
if 'records' not in globals():
    SEQ_STORE = os.path.join(WORK_DIR, 'sequences')
    os.makedirs(SEQ_STORE, exist_ok=True)
    if os.path.exists(PROGRESS_CSV):
        _df = pd.read_csv(PROGRESS_CSV)
        records    = _df.to_dict('records')
        done_parts = set(_df['part'].unique())
    else:
        records, done_parts = [], set()

# word_map=None: SYN은 JSON 내부에서 레이블 직접 추출
EXTRA_SOURCES = {
    'crowd_01': (39580, crowd_video_to_word),
    'crowd_02': (39582, crowd_video_to_word),
    'syn_01':   (39618, None),
}

for part_id, (filekey, word_map) in EXTRA_SOURCES.items():
    if part_id in done_parts: print(f'  {part_id} 완료 — 건너뜀'); continue

    print(f'\n[{part_id}] filekey={filekey}')
    if os.path.exists(TMP_DIR): shutil.rmtree(TMP_DIR)
    os.makedirs(TMP_DIR, exist_ok=True)

    zip_paths = aihub_download_and_get_zips(filekey, TMP_DIR)
    if not zip_paths: print('  zip 없음 — 건너뜀'); continue

    extract_dir = os.path.join(TMP_DIR, 'extract')
    os.makedirs(extract_dir, exist_ok=True)
    for zp in zip_paths:
        try:
            with zipfile.ZipFile(zp) as zf: zf.extractall(extract_dir)
        except zipfile.BadZipFile: print(f'  손상된 zip: {zp}')

    if word_map is not None:
        added = process_keypoint_dir(extract_dir, part_id, word_map, records, SEQ_STORE)
    else:
        # SYN: 키포인트 JSON 내 레이블 직접 파싱
        added = 0
        for path in tqdm(glob.glob(os.path.join(extract_dir,'**','*.json'), recursive=True),
                         desc=f'[{part_id}] SYN', leave=False):
            try:
                with open(path, encoding='utf-8') as f: data = json.load(f)
            except: continue
            word = extract_word_from_json(data)
            if word is None or word not in TARGET_SET: continue
            seq = sequence_from_keypoint_file(path)
            if seq is None: continue
            sample_id = f'{part_id}_{added}'
            np.save(os.path.join(SEQ_STORE, f'{sample_id}.npy'), seq)
            records.append({'sample_id': sample_id,
                            'video_id':  os.path.basename(path),
                            'word': word, 'part': part_id})
            added += 1

    done_parts.add(part_id)
    print(f'  +{added:,}개 (누적 {len(records):,})')
    pd.DataFrame(records).to_csv(PROGRESS_CSV, index=False)
    shutil.rmtree(TMP_DIR, ignore_errors=True)

meta_df = pd.DataFrame(records)
print(f'\n전체: {len(meta_df):,}개')
print(meta_df['word'].value_counts().to_string())
missing = [w for w in TARGET_WORDS if w not in meta_df['word'].values]
if missing: print(f'\n데이터 없는 단어: {missing}')

In [ ]:
# ── 셀 6. 데이터 분포 시각화 ───────────────────────────────────
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd

meta_df = pd.read_csv(PROGRESS_CSV)
counts  = meta_df['word'].value_counts()
ordered = [counts.get(w, 0) for w in TARGET_WORDS]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

bars = axes[0].bar(range(len(TARGET_WORDS)), ordered, color='#4C72B0')
axes[0].set_xticks(range(len(TARGET_WORDS)))
axes[0].set_xticklabels(TARGET_WORDS, rotation=60, ha='right')
axes[0].set_ylabel('샘플 수')
axes[0].set_title('단어별 샘플 수')
for bar, v in zip(bars, ordered):
    if v > 0: axes[0].text(bar.get_x()+bar.get_width()/2, v, str(v),
                            ha='center', va='bottom', fontsize=7)

cat_counts = Counter()
for w, c in counts.items(): cat_counts[WORD_CATEGORY.get(w, '기타')] += c
axes[1].pie(list(cat_counts.values()), labels=list(cat_counts.keys()),
            autopct='%1.1f%%', startangle=90, colors=plt.cm.Set2.colors)
axes[1].set_title('카테고리별 비율')

plt.tight_layout()
plt.show()
print(f'총 {len(meta_df):,}개 | 클래스 {len(counts)}개 | 평균 {len(meta_df)/len(counts):.0f}개/단어')

In [ ]:
# ── 셀 7. 데이터 전처리 ────────────────────────────────────────
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

meta_df   = pd.read_csv(PROGRESS_CSV)
SEQ_STORE = os.path.join(WORK_DIR, 'sequences')

# .npy 파일 로드
X_seq, y_raw = [], []
for row in meta_df.itertuples(index=False):
    p = os.path.join(SEQ_STORE, f'{row.sample_id}.npy')
    if not os.path.exists(p): continue
    X_seq.append(np.load(p))
    y_raw.append(row.word)

X_seq = np.stack(X_seq)   # (N, SEQ_LEN, FEATURE_DIM)
y_raw = np.array(y_raw)
print(f'시퀀스 데이터: {X_seq.shape}')

# 클래스별 최소 샘플 수 확인
unique, cnts = np.unique(y_raw, return_counts=True)
min_cnt = cnts.min()
print(f'클래스 수: {len(unique)}개 | 최솟값: {min_cnt}개 ({unique[cnts.argmin()]})')
if min_cnt < 10:
    print('경고: 샘플이 10개 미만인 클래스가 있습니다. 학습 품질에 영향을 줄 수 있습니다.')

# 레이블 인코딩
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
num_classes = len(label_encoder.classes_)

# sklearn용: 시퀀스 평균+표준편차 → 84차원 (시간 정보 요약)
X_flat = np.concatenate([X_seq.mean(axis=1), X_seq.std(axis=1)], axis=1)

# 70 / 10 / 20 분할 (stratify: 클래스 비율 유지)
X_tv_f, X_te_f, X_tv_s, X_te_s, y_tv, y_te = train_test_split(
    X_flat, X_seq, y, test_size=0.20, random_state=42, stratify=y)
X_tr_f, X_va_f, X_tr_s, X_va_s, y_tr, y_va = train_test_split(
    X_tv_f, X_tv_s, y_tv, test_size=0.125, random_state=42, stratify=y_tv)

print(f'train {len(y_tr):,} / val {len(y_va):,} / test {len(y_te):,}')
print(f'sklearn 특징 차원: {X_tr_f.shape[1]} | 시퀀스 shape: {X_tr_s.shape}')

In [ ]:
# ── 셀 8. 모델 학습 및 비교 ────────────────────────────────────
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

results = {}  # {name: {'acc', 'f1', 'time', 'model', 'type'}}

def train_sklearn(name, clf, X_tr, y_tr, X_te, y_te, model_type='sklearn'):
    t0 = time.time()
    clf.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    pred = clf.predict(X_te)
    results[name] = {
        'acc':   accuracy_score(y_te, pred),
        'f1':    f1_score(y_te, pred, average='macro'),
        'time':  elapsed,
        'model': clf,
        'type':  model_type,
    }
    print(f'{name:<22} acc={results[name]["acc"]:.4f}  f1={results[name]["f1"]:.4f}  {elapsed:.1f}s')

print('='*60)
print('정적 특징 기반 모델 (mean+std 84차원)')
print('='*60)

train_sklearn('RandomForest',
    RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    X_tr_f, y_tr, X_te_f, y_te)

# SVM: 샘플이 많으면 학습 시간 급증 → 5만 초과 시 skip
if len(X_tr_f) <= 50000:
    train_sklearn('SVM (RBF)',
        SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42),
        X_tr_f, y_tr, X_te_f, y_te)
else:
    print(f'SVM (RBF)              샘플 {len(X_tr_f):,}개 > 50000 — skip')

train_sklearn('MLP',
    MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=500,
                  early_stopping=True, random_state=42),
    X_tr_f, y_tr, X_te_f, y_te)

# HistGradientBoosting: 대용량에서도 빠른 GBM
train_sklearn('HistGradBoost',
    HistGradientBoostingClassifier(max_iter=200, max_depth=6,
                                   random_state=42),
    X_tr_f, y_tr, X_te_f, y_te)

print()
print('='*60)
print('시퀀스 기반 딥러닝 모델 (32×42)')
print('='*60)

# LSTM / GRU 공통 학습 함수
def train_dl(name, model, X_tr_s, y_tr, X_te_s, y_te, epochs=40, batch=64):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    Xt = torch.tensor(X_tr_s, dtype=torch.float32, device=DEVICE)
    yt = torch.tensor(y_tr,   dtype=torch.long,    device=DEVICE)
    Xv = torch.tensor(X_te_s, dtype=torch.float32, device=DEVICE)

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        # 매 에포크 무작위 순서로 배치
        perm = torch.randperm(len(Xt), device=DEVICE)
        for i in range(0, len(perm), batch):
            idx = perm[i:i+batch]
            optimizer.zero_grad()
            loss = criterion(model(Xt[idx]), yt[idx])
            loss.backward()
            optimizer.step()
        if (epoch+1) % 10 == 0:
            print(f'  {name} epoch {epoch+1}/{epochs}  loss={loss.item():.4f}')

    model.eval()
    with torch.no_grad():
        pred = model(Xv).argmax(1).cpu().numpy()
    elapsed = time.time() - t0
    results[name] = {
        'acc':   accuracy_score(y_te, pred),
        'f1':    f1_score(y_te, pred, average='macro'),
        'time':  elapsed,
        'model': model,
        'type':  name.lower().replace(' ','_'),
    }
    print(f'{name:<22} acc={results[name]["acc"]:.4f}  f1={results[name]["f1"]:.4f}  {elapsed:.1f}s')

class SignLSTM(nn.Module):
    def __init__(self, input_dim, hidden=128, layers=2, n_cls=30, dropout=0.3):
        super().__init__()
        self.rnn  = nn.LSTM(input_dim, hidden, layers, batch_first=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(hidden,128), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(128, n_cls))
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1])

class SignGRU(nn.Module):
    # GRU: LSTM보다 파라미터 적고 학습 빠름
    def __init__(self, input_dim, hidden=128, layers=2, n_cls=30, dropout=0.3):
        super().__init__()
        self.rnn  = nn.GRU(input_dim, hidden, layers, batch_first=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(hidden,128), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(128, n_cls))
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1])

train_dl('LSTM', SignLSTM(FEATURE_DIM, n_cls=num_classes),
         X_tr_s, y_tr, X_te_s, y_te)
train_dl('GRU',  SignGRU(FEATURE_DIM,  n_cls=num_classes),
         X_tr_s, y_tr, X_te_s, y_te)

# ── 비교 표 출력 ───────────────────────────────────────────────
print()
print('='*60)
print(f'{"모델":<22} {"정확도":>8} {"F1(macro)":>10} {"시간(s)":>9}')
print('-'*60)
for name, r in sorted(results.items(), key=lambda x: -x[1]['acc']):
    print(f'{name:<22} {r["acc"]:>8.4f} {r["f1"]:>10.4f} {r["time"]:>9.1f}')
print('='*60)

best_name  = max(results, key=lambda n: results[n]['acc'])
best_model = results[best_name]['model']
print(f'\n최고 모델: {best_name}  ({results[best_name]["acc"]*100:.1f}%)')

# ── 비교 바 차트 ───────────────────────────────────────────────
names = list(results.keys())
accs  = [results[n]['acc']*100 for n in names]
f1s   = [results[n]['f1']*100  for n in names]
# ML은 파란 계열, DL은 주황 계열
dl_set = {'LSTM', 'GRU'}
colors = ['#E67E22' if n in dl_set else '#2980B9' for n in names]

x = range(len(names))
fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar([i-0.2 for i in x], accs, 0.35, label='정확도(%)', color=colors, alpha=0.9)
bars2 = ax.bar([i+0.2 for i in x], f1s,  0.35, label='F1 macro(%)', color=colors, alpha=0.5)
ax.set_xticks(list(x)); ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylim(0, 115); ax.set_ylabel('점수 (%)')
ax.set_title('모델 성능 비교')
for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{bar.get_height():.1f}', ha='center', fontsize=8)
ml_p = mpatches.Patch(color='#2980B9', label='ML 모델')
dl_p = mpatches.Patch(color='#E67E22', label='DL 모델')
ax.legend(handles=[ml_p, dl_p, *ax.get_legend_handles_labels()[0]])
plt.tight_layout(); plt.show()

In [ ]:
# ── 셀 9. 검증 세트 평가 + 혼동 행렬 ─────────────────────────
# 주의: 셀 0(헬퍼), 셀 8(학습 결과) 실행 후 실행하세요.
import os, glob, json, zipfile, tarfile, shutil
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import (confusion_matrix, classification_report,
                              accuracy_score, f1_score)

VAL_DIR = '/content/sign_val'
if os.path.exists(VAL_DIR): shutil.rmtree(VAL_DIR)
os.makedirs(VAL_DIR, exist_ok=True)

# 검증 형태소 (filekey 39478) 다운로드
val_morph_dir = os.path.join(VAL_DIR, 'morph')
os.makedirs(val_morph_dir, exist_ok=True)
for zp in aihub_download_and_get_zips(39478, val_morph_dir):
    with zipfile.ZipFile(zp) as zf: zf.extractall(val_morph_dir)

val_video_to_word = {}
for path in glob.glob(os.path.join(val_morph_dir, '**', '*.json'), recursive=True):
    try:
        with open(path, encoding='utf-8') as f: data = json.load(f)
    except: continue
    word = extract_word_from_json(data)
    if word: val_video_to_word[video_id_from_path(path)] = word.strip()
print(f'검증 형태소: {len(val_video_to_word):,}개')

# 검증 키포인트 (filekey 39479) 다운로드
val_kp_dir = os.path.join(VAL_DIR, 'kp')
os.makedirs(val_kp_dir, exist_ok=True)
for zp in aihub_download_and_get_zips(39479, val_kp_dir):
    try:
        with zipfile.ZipFile(zp) as zf: zf.extractall(val_kp_dir)
    except zipfile.BadZipFile: print(f'손상된 zip: {zp}')

X_val, y_val_raw = [], []
for path in glob.glob(os.path.join(val_kp_dir, '**', '*.json'), recursive=True):
    vid  = video_id_from_path(path)
    word = val_video_to_word.get(vid)
    if word is None or word not in TARGET_SET: continue
    seq = sequence_from_keypoint_file(path)
    if seq is None: continue
    X_val.append(seq); y_val_raw.append(word)
print(f'검증 샘플: {len(X_val):,}개')

if len(X_val) == 0:
    print('검증 샘플 없음')
else:
    X_val = np.stack(X_val)
    keep  = [i for i, w in enumerate(y_val_raw) if w in set(label_encoder.classes_)]
    X_val = X_val[keep]
    y_val_true = label_encoder.transform([y_val_raw[i] for i in keep])

    print()
    print('='*55)
    print(f'{"모델":<22} {"검증 정확도":>10} {"F1":>8}')
    print('-'*55)
    for name, r in sorted(results.items(), key=lambda x: -x[1]['acc']):
        m = r['model']
        if r['type'] in ('lstm', 'gru'):
            m.eval()
            with torch.no_grad():
                xb   = torch.tensor(X_val, dtype=torch.float32, device=DEVICE)
                pred = m(xb).argmax(1).cpu().numpy()
        else:
            X_va_f = np.concatenate([X_val.mean(axis=1), X_val.std(axis=1)], axis=1)
            pred   = m.predict(X_va_f)
        v_acc = accuracy_score(y_val_true, pred)
        v_f1  = f1_score(y_val_true, pred, average='macro')
        print(f'{name:<22} {v_acc:>10.4f} {v_f1:>8.4f}')
    print('='*55)

    # 최고 모델 혼동 행렬
    m = results[best_name]['model']
    if results[best_name]['type'] in ('lstm', 'gru'):
        m.eval()
        with torch.no_grad():
            xb   = torch.tensor(X_val, dtype=torch.float32, device=DEVICE)
            pred = m(xb).argmax(1).cpu().numpy()
    else:
        X_va_f = np.concatenate([X_val.mean(axis=1), X_val.std(axis=1)], axis=1)
        pred   = m.predict(X_va_f)

    cm = confusion_matrix(y_val_true, pred, labels=range(num_classes))
    plt.figure(figsize=(13, 11))
    plt.imshow(cm, cmap='Blues')
    plt.colorbar()
    ticks = range(num_classes)
    plt.xticks(ticks, label_encoder.classes_, rotation=90)
    plt.yticks(ticks, label_encoder.classes_)
    plt.xlabel('예측'); plt.ylabel('실제')
    plt.title(f'혼동 행렬 — {best_name} (검증 세트)')
    plt.tight_layout(); plt.show()

    print(classification_report(y_val_true, pred,
          labels=range(num_classes), target_names=label_encoder.classes_,
          zero_division=0))

shutil.rmtree(VAL_DIR, ignore_errors=True)

In [ ]:
# ── 셀 10. 모델 저장 + 로컬 다운로드 ─────────────────────────
import os, json, pickle
import torch
from google.colab import files

SAVE_DIR = '/content/sign_model_out'
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PKL   = os.path.join(SAVE_DIR, 'sign_model.pkl')
ENCODER_PKL = os.path.join(SAVE_DIR, 'label_encoder.pkl')
META_JSON   = os.path.join(SAVE_DIR, 'model_meta.json')

model_type = results[best_name]['type']  # 'lstm', 'gru', 'sklearn'

if model_type in ('lstm', 'gru'):
    PT_PATH = os.path.join(SAVE_DIR, f'sign_{model_type}_state.pt')
    torch.save(best_model.state_dict(), PT_PATH)
    with open(MODEL_PKL, 'wb') as f:
        pickle.dump({'type': model_type,
                     'state_path': os.path.basename(PT_PATH)}, f)
else:
    with open(MODEL_PKL, 'wb') as f:
        pickle.dump({'type': 'sklearn', 'model': best_model}, f)

with open(ENCODER_PKL, 'wb') as f: pickle.dump(label_encoder, f)

# 전체 모델 성능도 저장 (추후 비교 참고용)
all_results_summary = {
    name: {'acc': r['acc'], 'f1': r['f1'], 'time': r['time']}
    for name, r in results.items()
}

meta = {
    'best_model':       best_name,
    'best_model_type':  model_type,
    'num_classes':      int(num_classes),
    'classes':          label_encoder.classes_.tolist(),
    'seq_len':          SEQ_LEN,
    'feature_dim':      FEATURE_DIM,
    'n_keypoints':      N_KEYPOINTS,
    'test_accuracy':    float(results[best_name]['acc']),
    'test_f1_macro':    float(results[best_name]['f1']),
    'all_models':       all_results_summary,
}
with open(META_JSON, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('저장 파일:')
for p in os.listdir(SAVE_DIR): print(f'  {p}')
print(f'\n최고 모델: {best_name}  정확도: {results[best_name]["acc"]*100:.1f}%')

files.download(MODEL_PKL)
files.download(ENCODER_PKL)
files.download(META_JSON)
if model_type in ('lstm', 'gru'):
    files.download(PT_PATH)